In [2]:
import torch
import torch.nn as nn

import sys

sys.path.append("..")

from src.paths import Paths
from src.config import TrainingConfig
from src.utils import get_device, initialize_resnet18
from src.data_manager import DataManager
from src.model_trainer import ModelTrainer

In [3]:
training_config = TrainingConfig()
training_config.epochs = 7
device = get_device()
dataset_name = "cifar10"

In [4]:
paths = Paths()
data_manager = DataManager(root=paths.DATA_DIR)

train_loader, val_loader, test_loader = data_manager.get_data_loaders(
    dataset_name=dataset_name,
    batch_size=training_config.batch_size,
    download=False,
)

class_names = data_manager.get_class_names(dataset_name)

print(f"Training data size: {len(train_loader) * 64}")
print(f"Validation data size: {len(val_loader) * 64}")
print(f"Test data size: {len(test_loader) * 64}")

/home/mikolajmalolepszy/Desktop/Projects/mgr/.venv/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Training data size: 45056
Validation data size: 5056
Test data size: 10048


In [5]:
model = initialize_resnet18()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=training_config.learning_rate,
)

In [6]:
trainer = ModelTrainer(
    model=model,
    device=device,
    criterion=criterion,
    optimizer=optimizer,
    class_names=class_names,
)

In [7]:
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=training_config.epochs,
)

Epoch 1/7
Train - Loss: 0.5704 Acc: 0.8066
Val   - Loss: 0.6995 Acc: 0.7728
-------------------------
Epoch 2/7
Train - Loss: 0.3301 Acc: 0.8877
Val   - Loss: 0.4023 Acc: 0.8640
-------------------------
Epoch 3/7
Train - Loss: 0.2308 Acc: 0.9202
Val   - Loss: 0.3787 Acc: 0.8740
-------------------------
Epoch 4/7
Train - Loss: 0.1787 Acc: 0.9385
Val   - Loss: 0.3558 Acc: 0.8904
-------------------------
Epoch 5/7
Train - Loss: 0.1226 Acc: 0.9573
Val   - Loss: 0.4155 Acc: 0.8728
-------------------------
Epoch 6/7
Train - Loss: 0.1061 Acc: 0.9634
Val   - Loss: 0.3171 Acc: 0.9034
-------------------------
Epoch 7/7
Train - Loss: 0.0852 Acc: 0.9703
Val   - Loss: 0.3844 Acc: 0.8942
-------------------------


In [12]:
results = trainer.evaluate(test_loader)

print(f"OVERALL ACCURACY: {results['accuracy']:.4f}\n\n")
print(results["classification_report"])

OVERALL ACCURACY: 0.8927


              precision    recall  f1-score   support

    airplane       0.90      0.92      0.91      1000
  automobile       0.97      0.94      0.96      1000
        bird       0.88      0.87      0.88      1000
         cat       0.81      0.79      0.80      1000
        deer       0.97      0.76      0.85      1000
         dog       0.79      0.88      0.83      1000
        frog       0.91      0.94      0.92      1000
       horse       0.88      0.94      0.91      1000
        ship       0.95      0.93      0.94      1000
       truck       0.90      0.97      0.93      1000

    accuracy                           0.89     10000
   macro avg       0.90      0.89      0.89     10000
weighted avg       0.90      0.89      0.89     10000



In [ ]:
trainer.save_model(paths.CIFAR10_MODEL)

/home/mikolajmalolepszy/Desktop/Projects/mgr/models/resnet18_cifar10.pth
